In [ ]:
# Runtime ▸ Change runtime type ▸ GPU  (T4/ A100)  BEFORE running this cell
!pip install -U timm==0.9.12 torch torchvision torchaudio scikit-learn tqdm albumentations --quiet

import torch, os, subprocess, sys
assert torch.cuda.is_available(), "⚠️ GPU not detected. Go to Runtime ▸ Change runtime type ▸ GPU."
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_DIR   = "/content/drive/MyDrive/PS-1/dataset"  # one folder, 14 sub-folders
VAL_RATIO  = 0.20                     # 80/20 split

NUM_CLASSES = 14
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 20
LR          = 3e-4
FREEZE_EPOCHS = 5    # freeze backbone for the first 5

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np, json, os, torch, random, cv2

IMG_SIZE = 224      # keep same as before
VAL_RATIO = 0.20

# ---- Albumentations pipeline (lots of variety → ≥4× effective data) ----
alb_train = A.Compose([
    A.SmallestMaxSize(max_size=IMG_SIZE+32),
    A.RandomCrop(width=IMG_SIZE, height=IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2,0.2,p=0.5),
    A.HueSaturationValue(10,15,10,p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10,
                       rotate_limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.CoarseDropout(max_holes=4, max_height=IMG_SIZE//8,
                    max_width=IMG_SIZE//8, p=0.3),
    A.ToFloat(max_value=255.0),
    ToTensorV2(),
])

alb_val = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.ToFloat(max_value=255.0),
    ToTensorV2(),
])

class AlbumentationsDataset(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root)
        self.alb_transform = transform
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)      # BGR
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.alb_transform(image=img)["image"]
        return img, label

full_ds_train = AlbumentationsDataset(DATA_DIR, alb_train)
full_ds_val   = AlbumentationsDataset(DATA_DIR, alb_val)

train_idx, val_idx = train_test_split(
    np.arange(len(full_ds_train)),
    test_size=VAL_RATIO,
    stratify=full_ds_train.targets,
    random_state=42)

train_ds = Subset(full_ds_train, train_idx)
val_ds   = Subset(full_ds_val,   val_idx)

json.dump(full_ds_train.class_to_idx, open("class_to_idx.json", "w"))

train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                      num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}")


/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-21-1951612928.py:20: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=4, max_height=IMG_SIZE//8,


Train: 11,385  Val: 2,847


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
import timm, torch.nn as nn, torch

model = timm.create_model(
    "mobilevitv2_050.cvnets_in1k",  # ~1.4 M params, fast
    pretrained=True,
    num_classes=NUM_CLASSES
).to(device)

# freeze everything except the classifier head at start
for n, p in model.named_parameters():
    if not n.startswith("head"):
        p.requires_grad = False     # will unfreeze later

In [ ]:
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-3)
sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler    = GradScaler()

hist = {"train_loss":[], "val_loss":[], "val_acc":[]}
best_acc   = 0
PATIENCE   = 3
bad_epochs = 0

for epoch in range(EPOCHS):
    # -------- unfreeze backbone after FREEZE_EPOCHS --------
    if epoch == FREEZE_EPOCHS:
        for p in model.parameters(): p.requires_grad = True
        opt = torch.optim.AdamW(model.parameters(), lr=LR/3, weight_decay=5e-3)
        print(f"🔓  Unfroze backbone at epoch {epoch}")

    # -------- training ------------------------------------
    model.train(); running_loss = 0
    for x,y in tqdm(train_dl, leave=False, desc=f"Train {epoch+1}/{EPOCHS}"):
        x,y = x.to(device,non_blocking=True), y.to(device,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast():
            loss = criterion(model(x),y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        running_loss += loss.item()*x.size(0)
    sched.step()

    # -------- validation ----------------------------------
    model.eval(); v_loss = 0; correct = total = 0
    with torch.no_grad(), autocast():
        for x,y in val_dl:
            x,y = x.to(device,non_blocking=True), y.to(device,non_blocking=True)
            out = model(x); v_loss += criterion(out,y).item()*x.size(0)
            preds = out.argmax(1); correct += (preds==y).sum().item(); total+=y.size(0)
    tr_loss = running_loss/len(train_ds)
    v_loss  = v_loss/len(val_ds)
    v_acc   = correct/total
    hist["train_loss"].append(tr_loss)
    hist["val_loss"].append(v_loss)
    hist["val_acc"].append(v_acc)
    print(f"Epoch {epoch+1:>2}  loss {tr_loss:.4f}  val_loss {v_loss:.4f}  val_acc {v_acc:.2%}")

    # -------- early-stopping criteria ---------------------
    if v_acc >= 0.99 or v_loss <= 0.05:
        print("🏁  Target reached — stopping.")
        torch.save(model.state_dict(), "best_model.pth")
        break
    if v_acc > best_acc + 1e-4:
        best_acc, bad_epochs = v_acc, 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("⏹️  Early stop (no improvement).")
            break

# -------- plot curves -------------------------------------
plt.figure(figsize=(6,3))
plt.subplot(1,2,1); plt.plot(hist["train_loss"], label="train"); plt.plot(hist["val_loss"], label="val")
plt.title("Loss"); plt.legend(); plt.xlabel("epoch")
plt.subplot(1,2,2); plt.plot(hist["val_acc"]); plt.title("Val accuracy"); plt.xlabel("epoch")
plt.tight_layout(); plt.show()


/tmp/ipython-input-23-3468942555.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()


Train 1/20:   0%|          | 0/178 [00:00<?, ?it/s]

/tmp/ipython-input-23-3468942555.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-23-3468942555.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Epoch  1  loss 2.5309  val_loss 2.4432  val_acc 62.07%


Train 2/20:   0%|          | 0/178 [00:00<?, ?it/s]

...

Train 10/20:   0%|          | 0/178 [00:00<?, ?it/s]

Epoch 10  loss 0.7276  val_loss 0.7716  val_acc 92.55%


Train 11/20:   0%|          | 0/178 [00:00<?, ?it/s]

Epoch 11  loss 0.6902  val_loss 0.7133  val_acc 94.98%


Train 12/20:   0%|          | 0/178 [00:00<?, ?it/s]

In [ ]:
torch.save(model.state_dict(), "mobilevitv2_050_gestures.pth")
print("Model saved ✅")

Model saved ✅
